# Staged regression estimators of small-OFF added value (48h morphological)

Within a cortical recording, bin NREM (or Wake) time into fixed epochs and measure, per
epoch, mean log delta power (`y`), the total area of medium/large OFF periods starting
in that epoch (`x1`), and the total area of small OFF periods (`x2`). Large OFF periods
are strongly associated with delta power. Give `x1` first claim on every scrap of delta
variance it can explain: is there signal left over that `x2` picks up?

Three staged estimators answer that. They differ only in how the predictor entering the
final regression is scaled. Everything below runs on standardized (z-scored, `ddof=0`)
`y`, `x1`, `x2` within each recording; let `r12 = corr(x1, x2)`.

### Estimator A: residualize the response only

```
step 1:  y   ~ x1     ->  e_y = residuals
step 2:  e_y ~ x2     ->  beta_A = slope on x2
```

Strip from delta everything medium/large OFF area predicts, then ask whether raw
small-OFF area tracks what remains.

### Estimator B: Frisch-Waugh-Lovell, residualizing both sides

```
step 1:  y    ~ x1      ->  e_y  = residuals
step 2:  x2   ~ x1      ->  e_x2 = residuals
step 3:  e_y  ~ e_x2    ->  beta_B = slope on e_x2
```

The same thing, except the predictor is the excess small-OFF area: how much more (or
less) small-OFF area an epoch had than its medium/large area would lead you to expect.

### Estimator C: FWL with the residualized predictor rescaled to unit SD

```
step 1:  y    ~ x1                ->  e_y   = residuals
step 2:  x2   ~ x1                ->  e_x2  = residuals
step 3:  e_x2n = e_x2 / sd(e_x2)      rescale to unit standard deviation
step 4:  e_y  ~ e_x2n             ->  beta_C = slope on e_x2n
```

`e_x2` has variance `1 - r12**2 < 1`, so a coefficient on it is not expressed per one
standard deviation of the regressor actually entering the fit; rescaling before fitting
restores that. With `y` standardized, `beta_C` is the semipartial (part) correlation
`sr` of `y` with `x2` given `x1`, and `beta_C**2`, the squared semipartial, is the
unique variance share. The two are routinely conflated, so they are kept distinct here.

### How they relate

Because OLS residuals are orthogonal to their regressor, `cov(e_y, x2) =
cov(e_y, e_x2)`: all three numerators agree. Only the denominators differ:
`var(x2) = 1`, `var(e_x2) = 1 - r12**2`, `var(e_x2n) = 1`, so

```
beta_A = beta_B * (1 - r12**2)
beta_C = beta_B * sqrt(1 - r12**2)
beta_A = beta_C * sqrt(1 - r12**2)
```

hence `|beta_A| <= |beta_C| <= |beta_B|`, with equality only at `r12 = 0`. All three
always carry the same sign. Estimator B is also algebraically identical to the
coefficient on `x2` in the joint fit `y ~ x1 + x2`, which is the Frisch-Waugh-Lovell
theorem. The joint model is fit here too, as an internal arithmetic check, and every
identity is verified numerically in the Validation section.

The three estimates differ by a monotone function of `r12` alone, so they carry no
independent information about the data.

### Estimator C is not scale-invariant; A and B are

The explicit `sd()` in step 3 does not cancel, with two consequences respected below:

- `y` must be z-scored, or the identity `beta_C == sr` fails, returning `sr * sd(y)`
  instead.
- The `ddof` used to z-score `x2` and the `ddof` inside `sd(e_x2)` must match. A
  `ddof=0`/`ddof=1` mismatch injects a relative error of `sqrt(n/(n-1))`, about
  `1.4e-4` at `n = 3500`, which fails a `1e-10` check. This notebook uses `ddof=0`
  throughout (`np.std` / `np.cov(..., ddof=0)` defaults). Checks 1-3 are scale-free and
  cannot be broken this way; checks 6-8 are what catch a botched Estimator C.

### Where HAC belongs

Newey-West affects standard errors only, never point estimates. Stage-1 regressions are
fit with plain OLS, since only their residuals matter; `cov_type="HAC"` is applied to
the final stage of each estimator and to the joint check model. Applying HAC everywhere
yields identical coefficients and only wastes time.

### Generated-regressor caveat

All three estimators feed stage-1 residuals into a later regression as if they were
observed data, so their standard errors do not propagate stage-1 estimation error. With
thousands of epochs per recording this should be negligible, but the standard errors
reported here are not exact on that account.

### What gets reported

One number, one p-value: `beta_B`, the conventional standardized partial coefficient,
read as "change in `y` per SD of `x2`, with `x1` held fixed". The closing Reporting
convention section says why, and when `beta_C` is the better choice instead.


In [ ]:
import pathlib

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pubplots as pp
import scipy.stats
import statsmodels.api as sm
import xarray as xr

from cnpix_local_sleep import files, hyp

In [ ]:
# ---- config ----
# Reuse the categorized 48h morphological cortical OFFs built by static_added_value.ipynb
# with state_mode="direct_48h". If absent, run that notebook first.
CACHE_PARQUET = pathlib.Path("./outputs/static_added_value/cache/offs_direct_48h.parquet")
OUTPUT_ROOT = pathlib.Path("./outputs/sequential_added_value")
save_plots = True

GROUP_COLS = ["subject", "probe", "structure"]
STATES = ["NREM", "Wake"]

# In the cached frame the categories are DISJOINT tiers, not nested sets:
# "BLAS" = BLAS; "CLAS" = CLAS-but-not-BLAS; "LLAS" = LLAS-but-not-CLAS.
# Summing areas across tiers is therefore legitimate and non-double-counting.
X1_TIERS = ["BLAS", "CLAS"]  # x1 "Medium/Large" = blas_area + clas_excl_area
X2_TIERS = ["LLAS"]          # x2 "Small"        = llas_excl_area
X1_LABEL = "Medium/Large"
X2_LABEL = "Small"

EPOCH_DURATION = 10.0            # s (primary)
EPOCH_SWEEP = [4.0, 10.0, 30.0]  # sensitivity
MIN_STATE_FRAC = 0.8             # keep an epoch only if >=80% of its samples are clean STATE
MIN_EPOCHS = 50                  # per-group minimum
HAC_TARGET_S = 300.0             # Newey-West window target (~5 min of autocorrelation)

REPORTED = "B"  # the single estimator quoted downstream (see the closing section)

TOL = 1e-10     # tolerance for the algebraic identity checks
INEQ_SLACK = 1e-12  # slack for the |A| <= |C| <= |B| ordering (equality holds at r12=0)


def maxlags_for(epoch_duration):
    """Newey-West lag count covering ~HAC_TARGET_S of epochs (30 at 10 s)."""
    return max(1, int(np.ceil(HAC_TARGET_S / epoch_duration)))


for state in STATES:
    (OUTPUT_ROOT / state).mkdir(parents=True, exist_ok=True)
print(f"epoch = {EPOCH_DURATION:.0f}s -> maxlags = {maxlags_for(EPOCH_DURATION)}; "
      f"reported estimator = {REPORTED}")

## Input OFFs

29 `(subject, probe, structure)` groups. The notebook runs both states in one
top-to-bottom pass; each state is filtered from the same cached frame.


In [ ]:
offs = pd.read_parquet(
    CACHE_PARQUET,
    columns=GROUP_COLS + ["start_time", "area", "category", "state"],
)
n_groups = offs.groupby(GROUP_COLS, observed=True).ngroups
print(f"{len(offs):,} cortical OFFs across {n_groups} groups")
print("state:   ", offs["state"].value_counts().to_dict())
print("category:", offs["category"].value_counts().to_dict())

state_offs = {s: offs[offs["state"] == s].reset_index(drop=True) for s in STATES}
for s in STATES:
    d = state_offs[s]
    print(f"{s:5s}: {len(d):>9,} OFFs, "
          f"{d.groupby(GROUP_COLS, observed=True).ngroups} groups, "
          f"{d['category'].value_counts().to_dict()}")

## Delta loader

The outcome is delta over all clean-state samples in an epoch, never restricted to OFF
intervals, so the OFF-area predictors cannot be definitionally coupled to it.

Each structure's band-power zarr is read from NFS once and masked for both states in
the same pass, then cached per `(subject, probe, structure, state)`. These loads
dominate runtime and are reused across the epoch-duration sweep, which therefore never
re-reads NFS.


In [ ]:
_DELTA_CACHE = {}


def load_group_delta(subject, probe, structure, state):
    """Return (time, log10_delta, fs) for clean-`state`, finite delta samples."""
    key = (subject, probe, structure, state)
    if key in _DELTA_CACHE:
        return _DELTA_CACHE[key]
    da = xr.load_dataarray(
        files.get_structure_bandpower_path(subject, probe, structure, "delta", True, "inst")
    )
    t = da["time"].values
    v = da.values
    fs = 1.0 / np.median(np.diff(t[:10000]))  # underlying sampling rate
    hg = hyp.load_statistical_condition_hypnograms(subject, probe)["Full.Conservative"]
    finite = np.isfinite(v)
    for st in STATES:  # one NFS read serves every state
        keep = hg.keep_states([st]).covers_time(t) & finite
        _DELTA_CACHE[(subject, probe, structure, st)] = (t[keep], np.log10(v[keep]), fs)
    return _DELTA_CACHE[key]

## Epoch table

Per group and epoch length: bin clean-state time into fixed epochs, take the epoch mean
of `log10(delta)` over that epoch's in-state samples, and accumulate OFF `area` per
predictor by each OFF's `start_time`. Epochs are kept only when at least
`MIN_STATE_FRAC` of their sample slots carry clean-state delta, which drops epochs
straddling a state boundary.

In [ ]:
def build_epoch_table(group_offs, t, logd, fs, epoch_duration):
    """Epoch table with columns x1 (medium/large area), x2 (small area), y (mean log delta)."""
    if t.size == 0:
        return pd.DataFrame()
    edges = np.arange(t[0], t[-1] + epoch_duration, epoch_duration)
    n = len(edges) - 1
    if n < 1:
        return pd.DataFrame()

    # Outcome: mean log-delta per epoch over clean-state samples.
    s_ep = np.clip(np.searchsorted(edges, t, side="right") - 1, 0, n - 1)
    dsum = np.zeros(n)
    dcnt = np.zeros(n)
    np.add.at(dsum, s_ep, logd)
    np.add.at(dcnt, s_ep, 1)
    valid = dcnt >= (MIN_STATE_FRAC * epoch_duration * fs)
    y = np.full(n, np.nan)
    y[valid] = dsum[valid] / dcnt[valid]

    # Predictors: summed OFF area of the disjoint tiers making up each predictor.
    cols = {}
    for name, tiers in [("x1", X1_TIERS), ("x2", X2_TIERS)]:
        sub = group_offs[group_offs["category"].isin(tiers)]
        area = np.zeros(n)
        if len(sub):
            oe = np.searchsorted(edges, sub["start_time"].to_numpy(), side="right") - 1
            ok = (oe >= 0) & (oe < n)
            np.add.at(area, oe[ok], sub["area"].to_numpy()[ok].astype(float))
        cols[name] = area

    out = pd.DataFrame({"x1": cols["x1"], "x2": cols["x2"], "y": y})
    out["epoch_start"] = edges[:-1]
    return out.loc[valid].reset_index(drop=True)

## Per-group fitting

All three estimators, the joint check model, an independently computed semipartial
correlation `sr` (for validation check 6, deliberately not read off Estimator C), and
the mirror direction (`x1` residualized on `x2`, so the medium/large unique coefficient
is reported on the same footing) are fit from one z-scored epoch table.

`r12` is computed on exactly the rows the regressions see, after the `NaN`-drop;
computing it on any other row set is the usual cause of a failing shrinkage identity.
Every standard deviation here uses `ddof=0`, matching the z-scoring, as Estimator C
requires.


In [ ]:
def zscore(a):
    a = np.asarray(a, float)
    sd = a.std(ddof=0)
    return (a - a.mean()) / sd if sd > 0 else np.zeros_like(a)


def resid(a, b):
    """Residualize a on b with plain OLS (point estimates only -- no HAC needed)."""
    return sm.OLS(a, sm.add_constant(b)).fit().resid


def _staged(y, e_y, x_raw, e_x, hac):
    """The three staged fits for one direction, sharing stage-1 residuals ``e_y``.

    ``x_raw`` is the z-scored predictor of interest, ``e_x`` its residual on the other
    predictor. Returns (fitA, fitB, fitC, e_xn, sr) where ``sr`` is the semipartial
    correlation computed independently of fitC (validation check 6).
    """
    e_xn = e_x / e_x.std(ddof=0)  # ddof matches the z-scoring -- see the prose above
    fitA = sm.OLS(e_y, sm.add_constant(x_raw)).fit(**hac)
    fitB = sm.OLS(e_y, sm.add_constant(e_x)).fit(**hac)
    fitC = sm.OLS(e_y, sm.add_constant(e_xn)).fit(**hac)
    sr = float(np.cov(y, e_xn, ddof=0)[0, 1] / (y.std(ddof=0) * e_xn.std(ddof=0)))
    return fitA, fitB, fitC, e_xn, sr


def fit_group(epoch_df, epoch_duration):
    """Sequential estimators A, B and C (+ mirror direction and joint check model).

    Returns None if too few epochs survive or either predictor is constant.
    """
    base = epoch_df[["x1", "x2", "y"]].dropna()
    if len(base) < MIN_EPOCHS:
        return None
    x1 = zscore(base["x1"].to_numpy())
    x2 = zscore(base["x2"].to_numpy())
    y = zscore(base["y"].to_numpy())
    if x1.std(ddof=0) == 0 or x2.std(ddof=0) == 0:
        return None

    hac = dict(cov_type="HAC", cov_kwds={"maxlags": maxlags_for(epoch_duration)})
    r12 = float(np.corrcoef(x1, x2)[0, 1])

    # --- small-OFF direction: residualize on x1 (medium/large) ---
    e_y = resid(y, x1)     # stage 1, plain OLS
    e_x2 = resid(x2, x1)   # stage 2 (Estimators B and C), plain OLS
    fitA, fitB, fitC, e_x2n, sr = _staged(y, e_y, x2, e_x2, hac)

    # --- mirror direction: residualize on x2 (small) ---
    e_y_m = resid(y, x2)
    e_x1 = resid(x1, x2)
    mA, mB, mC, _, sr_m = _staged(y, e_y_m, x1, e_x1, hac)

    # --- joint check model: y ~ x1 + x2 ---
    joint = sm.OLS(y, sm.add_constant(np.column_stack([x1, x2]))).fit(**hac)

    return {
        "n_epochs": len(base),
        "maxlags": maxlags_for(epoch_duration),
        "r12": r12,
        "one_minus_r12sq": 1.0 - r12**2,
        "sqrt_one_minus_r12sq": np.sqrt(1.0 - r12**2),
        "beta_A": fitA.params[1], "se_A": fitA.bse[1], "p_A": fitA.pvalues[1],
        "beta_B": fitB.params[1], "se_B": fitB.bse[1], "p_B": fitB.pvalues[1],
        "beta_C": fitC.params[1], "se_C": fitC.bse[1], "p_C": fitC.pvalues[1],
        "sr": sr,
        "beta_joint": joint.params[2], "se_joint": joint.bse[2], "p_joint": joint.pvalues[2],
        "ratio_AB": fitA.params[1] / fitB.params[1],
        "ratio_CB": fitC.params[1] / fitB.params[1],
        # mirror direction (medium/large unique coefficient)
        "beta_A_medl": mA.params[1], "se_A_medl": mA.bse[1], "p_A_medl": mA.pvalues[1],
        "beta_B_medl": mB.params[1], "se_B_medl": mB.bse[1], "p_B_medl": mB.pvalues[1],
        "beta_C_medl": mC.params[1], "se_C_medl": mC.bse[1], "p_C_medl": mC.pvalues[1],
        "sr_medl": sr_m,
        "beta_joint_medl": joint.params[1], "se_joint_medl": joint.bse[1],
        "ratio_AB_medl": mA.params[1] / mB.params[1],
        "ratio_CB_medl": mC.params[1] / mB.params[1],
        # orthogonality diagnostics (validation checks 4 and 5)
        "corr_ex2_x1": float(np.corrcoef(e_x2, x1)[0, 1]),
        "corr_ey_x1": float(np.corrcoef(e_y, x1)[0, 1]),
        # marginal correlations and joint fit, for checks 8 and context
        "r_y1": float(np.corrcoef(y, x1)[0, 1]),
        "r_y2": float(np.corrcoef(y, x2)[0, 1]),
        "R2_joint": joint.rsquared,
    }

## Run both states at the primary epoch length

In [ ]:
def run_state(state, epoch_duration):
    """Fit every group of one state; returns (group_df, skipped)."""
    recs, skipped = [], []
    for keys, grp in state_offs[state].groupby(GROUP_COLS, observed=True):
        t, logd, fs = load_group_delta(*keys, state)
        et = build_epoch_table(grp, t, logd, fs, epoch_duration)
        res = fit_group(et, epoch_duration)
        if res is None:
            skipped.append((keys, len(et)))
            continue
        res.update(dict(zip(GROUP_COLS, keys)))
        recs.append(res)
    return pd.DataFrame(recs), skipped


group_dfs = {}
for state in STATES:
    gdf, skipped = run_state(state, EPOCH_DURATION)
    group_dfs[state] = gdf
    print(f"{state}: fitted {len(gdf)} groups "
          f"(median {gdf['n_epochs'].median():,.0f} epochs/group); skipped {len(skipped)}")
    for keys, n in skipped:  # no silent drops
        print(f"    skipped {keys}: {n} surviving epochs (<{MIN_EPOCHS} or degenerate predictor)")

display(
    pd.concat([group_dfs[s].assign(state=s) for s in STATES])[
        ["state"] + GROUP_COLS + ["n_epochs", "r12", "beta_A", "beta_C", "beta_B",
                                  "se_B", "p_B", "beta_joint", "ratio_AB", "ratio_CB"]
    ]
    .style.format({"n_epochs": "{:,}", "r12": "{:+.3f}", "beta_A": "{:+.4f}",
                   "beta_C": "{:+.4f}", "beta_B": "{:+.4f}", "se_B": "{:.4f}",
                   "p_B": "{:.2e}", "beta_joint": "{:+.4f}",
                   "ratio_AB": "{:.4f}", "ratio_CB": "{:.4f}"})
    .set_caption(f"Per-group sequential estimates ({X2_LABEL} OFF area | {X1_LABEL}), "
                 f"epoch = {EPOCH_DURATION:.0f}s, HAC SEs "
                 f"(|beta_A| <= |beta_C| <= |beta_B| by construction)")
)

## Validation

| # | check | tolerance |
|---|---|---|
| 1 | `beta_B` == joint model's coefficient on `x2` (FWL) | `< 1e-10` |
| 2 | `beta_A / beta_B` == `1 - r12**2` | `< 1e-10` |
| 3 | `sign(beta_A) == sign(beta_B) == sign(beta_C)` | exact |
| 4 | `corr(e_x2, x1)` ~ 0 (residualization actually worked) | `< 1e-10` |
| 5 | `corr(e_y, x1)` ~ 0 | `< 1e-10` |
| 6 | `beta_C` == independently computed `sr` | `< 1e-10` |
| 7 | `beta_C / beta_B` == `sqrt(1 - r12**2)` | `< 1e-10` |
| 8 | `beta_C**2` == `R2_joint - corr(y, x1)**2` (unique variance) | `< 1e-10` |
| 9 | `abs(beta_A) <= abs(beta_C) <= abs(beta_B)` | exact |

Max absolute errors are printed instead of bare `assert`s, so the notebook shows how
tightly the identities hold, per state and across all groups.

- Checks 2 and 7 are invariant to any affine rescaling of `y`, `x1`, `x2`, so they hold
  whether or not the columns are standardized. If either fails while check 1 passes, the
  cause is almost always that `r12` was computed on a different set of rows than the
  regression saw.
- Checks 6-8 are the ones that catch a botched Estimator C. A failure at roughly
  `1.4e-4` relative is the `ddof=0`/`ddof=1` mismatch described at the top, not a
  conceptual error.
- Checks 3 and 9 are counted as violations rather than measured as errors. The ordering
  in check 9 holds with equality at `r12 = 0`, so it is tested with a `1e-12` slack
  against floating-point noise.


In [ ]:
def identity_checks(gdf):
    """Max-abs-error (or violation count) for each algebraic identity, one state."""
    sign_A, sign_B, sign_C = np.sign(gdf["beta_A"]), np.sign(gdf["beta_B"]), np.sign(gdf["beta_C"])
    sign_mismatch = int(((sign_A != sign_B) | (sign_B != sign_C)).sum())
    aA, aB, aC = gdf["beta_A"].abs(), gdf["beta_B"].abs(), gdf["beta_C"].abs()
    order_violations = int(((aA > aC + INEQ_SLACK) | (aC > aB + INEQ_SLACK)).sum())
    rows = [
        dict(check="1", kind="max_abs_error",
             description="beta_B == joint coefficient on x2 (FWL)",
             value=float(np.abs(gdf["beta_B"] - gdf["beta_joint"]).max())),
        dict(check="2", kind="max_abs_error",
             description="beta_A / beta_B == 1 - r12^2",
             value=float(np.abs(gdf["ratio_AB"] - gdf["one_minus_r12sq"]).max())),
        dict(check="3", kind="violations",
             description="sign(beta_A) == sign(beta_B) == sign(beta_C)",
             value=float(sign_mismatch)),
        dict(check="4", kind="max_abs_error", description="corr(e_x2, x1) == 0",
             value=float(np.abs(gdf["corr_ex2_x1"]).max())),
        dict(check="5", kind="max_abs_error", description="corr(e_y, x1) == 0",
             value=float(np.abs(gdf["corr_ey_x1"]).max())),
        dict(check="6", kind="max_abs_error",
             description="beta_C == independently computed sr",
             value=float(np.abs(gdf["beta_C"] - gdf["sr"]).max())),
        dict(check="7", kind="max_abs_error",
             description="beta_C / beta_B == sqrt(1 - r12^2)",
             value=float(np.abs(gdf["ratio_CB"] - gdf["sqrt_one_minus_r12sq"]).max())),
        dict(check="8", kind="max_abs_error",
             description="beta_C^2 == R2_joint - corr(y, x1)^2  (unique variance)",
             value=float(np.abs(gdf["beta_C"] ** 2 - (gdf["R2_joint"] - gdf["r_y1"] ** 2)).max())),
        dict(check="9", kind="violations",
             description="abs(beta_A) <= abs(beta_C) <= abs(beta_B)",
             value=float(order_violations)),
        # ratio-free forms of 2 and 7 (robust when beta_B is near zero), and the mirror
        dict(check="2b", kind="max_abs_error",
             description="beta_A == beta_B * (1 - r12^2)  [ratio-free form]",
             value=float(np.abs(gdf["beta_A"] - gdf["beta_B"] * gdf["one_minus_r12sq"]).max())),
        dict(check="7b", kind="max_abs_error",
             description="beta_C == beta_B * sqrt(1 - r12^2)  [ratio-free form]",
             value=float(np.abs(gdf["beta_C"] - gdf["beta_B"] * gdf["sqrt_one_minus_r12sq"]).max())),
        dict(check="1m", kind="max_abs_error",
             description="mirror: beta_B_medl == joint coefficient on x1",
             value=float(np.abs(gdf["beta_B_medl"] - gdf["beta_joint_medl"]).max())),
        dict(check="2m", kind="max_abs_error",
             description="mirror: beta_A_medl / beta_B_medl == 1 - r12^2",
             value=float(np.abs(gdf["ratio_AB_medl"] - gdf["one_minus_r12sq"]).max())),
        dict(check="7m", kind="max_abs_error",
             description="mirror: beta_C_medl / beta_B_medl == sqrt(1 - r12^2)",
             value=float(np.abs(gdf["ratio_CB_medl"] - gdf["sqrt_one_minus_r12sq"]).max())),
        dict(check="8m", kind="max_abs_error",
             description="mirror: beta_C_medl^2 == R2_joint - corr(y, x2)^2",
             value=float(np.abs(gdf["beta_C_medl"] ** 2
                                - (gdf["R2_joint"] - gdf["r_y2"] ** 2)).max())),
    ]
    out = pd.DataFrame(rows)
    out["tolerance"] = np.where(out["kind"] == "violations", 0.0, TOL)
    out["passed"] = np.where(out["kind"] == "violations",
                             out["value"] == 0, out["value"] < TOL)
    out["k_groups"] = len(gdf)
    return out.rename(columns={"value": "max_abs_error_or_violations"})


check_dfs = {}
for state in STATES:
    cdf = identity_checks(group_dfs[state])
    check_dfs[state] = cdf
    print(f"--- {state} (k = {len(group_dfs[state])} groups) ---")
    for _, r in cdf.iterrows():
        flag = "OK " if r["passed"] else "FAIL"
        val = r["max_abs_error_or_violations"]
        shown = f"{int(val)} violations" if r["kind"] == "violations" else f"max|err| = {val:.3e}"
        print(f"  [{flag}] check {r['check']:>2s}: {shown:>22s}  {r['description']}")
    print()

all_passed = all(cdf["passed"].all() for cdf in check_dfs.values())
print(f"all identity checks pass (tolerance {TOL:g}; ordering slack {INEQ_SLACK:g}): {all_passed}")

## Pooling across recordings

Per-group `(beta, se)` are pooled with DerSimonian-Laird random effects. Raw epochs are
not pooled: recordings differ in length by several-fold and would dominate by epoch
count.

The three estimators' p-values are near-identical by construction. Within a recording
the coefficients differ by a positive factor, so they test the same null.


In [ ]:
def random_effects_meta(effects, variances):
    """DerSimonian-Laird random-effects pooling of generic (effect, variance)."""
    eff, v = np.asarray(effects, float), np.asarray(variances, float)
    w = 1.0 / v
    fe = np.sum(w * eff) / np.sum(w)
    q = np.sum(w * (eff - fe) ** 2)
    k = len(eff)
    c = np.sum(w) - np.sum(w**2) / np.sum(w)
    tau2 = max(0.0, (q - (k - 1)) / c) if c > 0 else 0.0
    wre = 1.0 / (v + tau2)
    pooled = np.sum(wre * eff) / np.sum(wre)
    se = 1.0 / np.sqrt(np.sum(wre))
    i2 = max(0.0, (q - (k - 1)) / q) * 100 if q > 0 else 0.0
    p = 2 * (1 - scipy.stats.norm.cdf(abs(pooled / se)))
    return dict(pooled=pooled, se=se, ci_lo=pooled - 1.96 * se,
                ci_hi=pooled + 1.96 * se, p=p, tau2=tau2, i_squared=i2, k=k)


# (estimator label, target predictor, beta column, se column)
ESTIMATORS = [
    ("A", X2_LABEL, "beta_A", "se_A"),
    ("B", X2_LABEL, "beta_B", "se_B"),
    ("C", X2_LABEL, "beta_C", "se_C"),
    ("A", X1_LABEL, "beta_A_medl", "se_A_medl"),
    ("B", X1_LABEL, "beta_B_medl", "se_B_medl"),
    ("C", X1_LABEL, "beta_C_medl", "se_C_medl"),
]


def pool_estimators(gdf, estimators=ESTIMATORS):
    rows = []
    for label, target, b, s in estimators:
        m = random_effects_meta(gdf[b], gdf[s] ** 2)
        rows.append(dict(estimator=label, target=target, reported=(label == REPORTED), **m))
    return pd.DataFrame(rows)


pooled_dfs = {}
for state in STATES:
    pdf = pool_estimators(group_dfs[state])
    pooled_dfs[state] = pdf
    print(f"--- {state} ---")
    for _, r in pdf.iterrows():
        tag = "  <- reported" if r["reported"] and r["target"] == X2_LABEL else ""
        print(f"  Estimator {r['estimator']}  {r['target']:12s}: pooled = {r['pooled']:+.4f} "
              f"[{r['ci_lo']:+.4f}, {r['ci_hi']:+.4f}]  p = {r['p']:.2e}  "
              f"I2 = {r['i_squared']:.0f}%  k = {int(r['k'])}{tag}")
    print()

display(
    pd.concat([pooled_dfs[s].assign(state=s) for s in STATES])[
        ["state", "estimator", "target", "pooled", "se", "ci_lo", "ci_hi", "p",
         "tau2", "i_squared", "k", "reported"]
    ]
    .style.format({"pooled": "{:+.4f}", "se": "{:.4f}", "ci_lo": "{:+.4f}",
                   "ci_hi": "{:+.4f}", "p": "{:.2e}", "tau2": "{:.4f}",
                   "i_squared": "{:.0f}%"})
    .set_caption("DerSimonian-Laird pooled sequential estimates, by state and estimator "
                 "(A, B, C are one finding under three scalings)")
)

### Shrinkage factors and the squared semipartial

`1 - r12**2` (Estimator A vs B) and `sqrt(1 - r12**2)` (Estimator C vs B) are the only
things separating the three estimates. `beta_C**2` is the unique variance share of
small-OFF area; the signed `beta_C` is kept alongside it, since squaring discards the
sign and would make a negative and a positive association of equal strength
indistinguishable.


In [ ]:
for state in STATES:
    g = group_dfs[state]
    print(f"{state:5s}: r12 in [{g['r12'].min():+.3f}, {g['r12'].max():+.3f}], "
          f"median {g['r12'].median():+.3f}")
    print(f"        A/B shrinkage 1-r12^2       median {g['one_minus_r12sq'].median():.3f} "
          f"(range {g['one_minus_r12sq'].min():.3f}-{g['one_minus_r12sq'].max():.3f})")
    print(f"        C/B shrinkage sqrt(1-r12^2) median {g['sqrt_one_minus_r12sq'].median():.3f} "
          f"(range {g['sqrt_one_minus_r12sq'].min():.3f}-{g['sqrt_one_minus_r12sq'].max():.3f})")
    print(f"        signed beta_C median {g['beta_C'].median():+.4f}; "
          f"squared semipartial beta_C^2 median {(g['beta_C'] ** 2).median():.5f} "
          f"(= {(g['beta_C'] ** 2).median() * 100:.3f}% of y variance, unique to {X2_LABEL})")

## Figures

Plotted under `pubplots.destination("figma")`: `figsize`, line widths and marker sizes go
through `pp.scale`, and no explicit font sizes are passed; text size is left to the
pubplots rcParams. Every figure is rendered once per state and written to that state's
output directory, so `outputs/sequential_added_value/<state>/` is self-contained.

In [ ]:
EST_COLORS = {"A": "#4c72b0", "B": "#c44e52", "C": "#dd8452"}


def save_fig(fig, state, name):
    if save_plots:
        fig.savefig(OUTPUT_ROOT / state / f"{name}.svg")


def forest(ax, gdf, effect_col, se_col, meta, xlabel):
    """Sorted per-group caterpillar with the random-effects pooled diamond."""
    order = np.argsort(gdf[effect_col].to_numpy())
    gd = gdf.iloc[order].reset_index(drop=True)
    labels = [" / ".join(str(r[c]) for c in GROUP_COLS) for _, r in gd.iterrows()]
    y = np.arange(len(gd))[::-1]
    ax.errorbar(gd[effect_col], y, xerr=1.96 * gd[se_col], fmt="o", color="steelblue",
                ecolor="steelblue", elinewidth=pp.scale(0.8), markersize=pp.scale(3),
                capsize=pp.scale(1.5))
    yo, hw = -1.5, 0.45
    ax.fill([meta["ci_lo"], meta["pooled"], meta["ci_hi"], meta["pooled"]],
            [yo, yo + hw, yo, yo - hw], color="firebrick", alpha=0.8)
    ax.axvline(0, color="grey", ls="--", lw=pp.scale(0.7))
    ax.set_yticks(list(y) + [yo])
    ax.set_yticklabels(labels + ["RE pooled"])
    ax.set_ylim(yo - 1, y[0] + 1)
    ax.set_xlabel(xlabel)


def pooled_row(state, estimator, target=None):
    target = target or X2_LABEL
    d = pooled_dfs[state]
    return d[(d["estimator"] == estimator) & (d["target"] == target)].iloc[0].to_dict()

### Figure 1: forest of per-group Estimator B (the reported small-OFF coefficient)

In [ ]:
for state in STATES:
    meta = pooled_row(state, REPORTED)
    with pp.destination("figma"):
        fig, ax = plt.subplots(figsize=pp.scale((4.2, 4.6)), constrained_layout=True)
        forest(ax, group_dfs[state], f"beta_{REPORTED}", f"se_{REPORTED}", meta,
               f"Estimator {REPORTED}: {X2_LABEL} OFF area | {X1_LABEL} (std. coef)")
        ax.set_title(f"{state} - pooled {meta['pooled']:+.3f} "
                     f"[{meta['ci_lo']:+.3f}, {meta['ci_hi']:+.3f}], "
                     f"I2 = {meta['i_squared']:.0f}%, k = {int(meta['k'])}")
        save_fig(fig, state, "forest_estimator_B")
        plt.show()

### Figure 2: estimator scatter, A vs B and C vs B

One point per group. The dashed line is the identity; the open crosses are the predicted
values (`beta_B * (1 - r12**2)` for A, `beta_B * sqrt(1 - r12**2)` for C), evaluated at
each group's own `r12`. The crosses land exactly on the observed points. Both panels
fall inside the identity line whenever `r12 != 0`, C less so than A, and neither crosses
zero.


In [ ]:
for state in STATES:
    gdf = group_dfs[state]
    panels = [("A", "beta_A", gdf["beta_B"] * gdf["one_minus_r12sq"],
               r"predicted  $\beta_B\,(1-r_{12}^2)$"),
              ("C", "beta_C", gdf["beta_B"] * gdf["sqrt_one_minus_r12sq"],
               r"predicted  $\beta_B\sqrt{1-r_{12}^2}$")]
    with pp.destination("figma"):
        fig, axes = plt.subplots(1, 2, figsize=pp.scale((5.6, 2.9)), constrained_layout=True)
        for ax, (est, col, pred, pred_label) in zip(axes, panels):
            ax.scatter(gdf["beta_B"], gdf[col], s=pp.scale(18), color=EST_COLORS[est],
                       alpha=0.85, zorder=3, label="observed")
            ax.scatter(gdf["beta_B"], pred, s=pp.scale(34), marker="x", color="black",
                       linewidths=pp.scale(0.9), zorder=4, label=pred_label)
            lims = [min(ax.get_xlim()[0], ax.get_ylim()[0]),
                    max(ax.get_xlim()[1], ax.get_ylim()[1])]
            ax.plot(lims, lims, "k--", lw=pp.scale(0.8), zorder=1, label="identity")
            ax.axhline(0, color="grey", lw=pp.scale(0.6), zorder=0)
            ax.axvline(0, color="grey", lw=pp.scale(0.6), zorder=0)
            ax.set_xlim(lims)
            ax.set_ylim(lims)
            ax.set_xlabel(r"Estimator B ($\beta_B$)")
            ax.set_ylabel(rf"Estimator {est} ($\beta_{est}$)")
            ax.legend(loc="best")
        fig.suptitle(f"{state} - staged estimators vs B ({X2_LABEL} | {X1_LABEL})")
        save_fig(fig, state, "estimator_scatter")
        plt.show()

### Figure 3: shrinkage vs collinearity

`ratio_AB = beta_A / beta_B` and `ratio_CB = beta_C / beta_B` against each group's
`r12`, over the analytic `1 - r12**2` and `sqrt(1 - r12**2)` curves. Every group sits on
its own curve by construction, and C sits above A everywhere; the spread along the
curves shows how much collinearity there actually is in these data.


In [ ]:
for state in STATES:
    gdf = group_dfs[state]
    rr = np.linspace(-1, 1, 401)
    with pp.destination("figma"):
        fig, ax = plt.subplots(figsize=pp.scale((3.2, 2.5)), constrained_layout=True)
        ax.plot(rr, np.sqrt(1 - rr**2), color=EST_COLORS["C"], lw=pp.scale(1.0), zorder=1,
                label=r"$\sqrt{1-r_{12}^2}$")
        ax.plot(rr, 1 - rr**2, color=EST_COLORS["A"], lw=pp.scale(1.0), zorder=1,
                label=r"$1-r_{12}^2$")
        ax.scatter(gdf["r12"], gdf["ratio_CB"], s=pp.scale(18), color=EST_COLORS["C"],
                   alpha=0.85, zorder=3, label=r"$\beta_C/\beta_B$")
        ax.scatter(gdf["r12"], gdf["ratio_AB"], s=pp.scale(18), color=EST_COLORS["A"],
                   alpha=0.85, zorder=3, label=r"$\beta_A/\beta_B$")
        ax.set_xlim(min(-0.05, gdf["r12"].min() - 0.1), max(0.05, gdf["r12"].max() + 0.1))
        ax.set_xlabel(r"$r_{12}=\mathrm{corr}(x_1,x_2)$")
        ax.set_ylabel("ratio to Estimator B")
        ax.set_title(f"{state} - shrinkage vs collinearity")
        ax.legend(loc="best")
        save_fig(fig, state, "shrinkage_vs_collinearity")
        plt.show()

### Epoch-duration sweep

Re-pool all three estimators at each epoch length in `EPOCH_SWEEP`, reusing the cached
delta series, so there are no further NFS reads. The `maxlags` HAC window is rescaled
with the epoch length so it always covers ~`HAC_TARGET_S`. The conclusion should not be
an artifact of the 10 s choice.


In [ ]:
sweep_rows = []
for ep in EPOCH_SWEEP:
    for state in STATES:
        gdf, skipped = run_state(state, ep)
        for keys, n in skipped:
            print(f"  [{state}, {ep:.0f}s] skipped {keys}: {n} epochs")
        pdf = pool_estimators(gdf)
        pdf.insert(0, "epoch_s", ep)
        pdf.insert(0, "state", state)
        sweep_rows.append(pdf)

sweep_df = pd.concat(sweep_rows, ignore_index=True)
display(
    sweep_df.query("target == @X2_LABEL")
    .assign(**{"pooled [95% CI]": lambda d: d.apply(
        lambda r: f"{r['pooled']:+.4f} [{r['ci_lo']:+.4f}, {r['ci_hi']:+.4f}]", axis=1)})
    .pivot(index=["state", "epoch_s"], columns="estimator", values="pooled [95% CI]")
    .style.set_caption(f"Pooled {X2_LABEL} coefficient by epoch length: Estimators A, B, C")
)

### Figure 4: pooled estimates across epoch lengths

In [ ]:
for state in STATES:
    sub = sweep_df.query("state == @state and target == @X2_LABEL")
    with pp.destination("figma"):
        fig, ax = plt.subplots(figsize=pp.scale((3.2, 2.3)), constrained_layout=True)
        for est, dx in [("A", -0.025), ("C", 0.0), ("B", 0.025)]:
            d = sub[sub["estimator"] == est].sort_values("epoch_s")
            x = np.log10(d["epoch_s"].to_numpy()) + dx
            ax.errorbar(x, d["pooled"], yerr=1.96 * d["se"], fmt="o-",
                        color=EST_COLORS[est], ecolor=EST_COLORS[est], lw=pp.scale(1.0),
                        elinewidth=pp.scale(0.9), markersize=pp.scale(4),
                        capsize=pp.scale(2), label=f"Estimator {est}")
        ax.axhline(0, color="grey", ls="--", lw=pp.scale(0.7))
        ax.set_xticks(np.log10(EPOCH_SWEEP))
        ax.set_xticklabels([f"{e:.0f}" for e in EPOCH_SWEEP])
        ax.set_xlabel("epoch duration (s)")
        ax.set_ylabel(f"pooled std. coef ({X2_LABEL} | {X1_LABEL})")
        ax.set_title(f"{state} - epoch-duration sweep")
        ax.legend(loc="best")
        save_fig(fig, state, "epoch_duration_sweep")
        plt.show()

### Figure 5: the three scalings on one axis

Pooled estimate and 95% CI for A, B and C, at the primary epoch length. The scaling
choice moves the number by a few percent (`1 - r12**2` / `sqrt(1 - r12**2)` at the
observed collinearity) and changes nothing about sign or significance. The reported
quantity is marked.


In [ ]:
for state in STATES:
    rows = [("A", pooled_row(state, "A")), ("C", pooled_row(state, "C")),
            ("B", pooled_row(state, "B"))]
    with pp.destination("figma"):
        fig, ax = plt.subplots(figsize=pp.scale((3.2, 1.5)), constrained_layout=True)
        for i, (est, m) in enumerate(rows):
            yy = len(rows) - 1 - i
            ax.errorbar(m["pooled"], yy,
                        xerr=[[m["pooled"] - m["ci_lo"]], [m["ci_hi"] - m["pooled"]]],
                        fmt="D" if est == REPORTED else "o", color=EST_COLORS[est],
                        ecolor=EST_COLORS[est], markersize=pp.scale(5),
                        elinewidth=pp.scale(1.0), capsize=pp.scale(2.5))
            ax.annotate(f"{m['pooled']:+.3f} [{m['ci_lo']:+.3f}, {m['ci_hi']:+.3f}]",
                        (m["pooled"], yy), textcoords="offset points",
                        xytext=(0, pp.scale(7)), ha="center")
        ax.axvline(0, color="grey", ls="--", lw=pp.scale(0.7))
        ax.set_yticks(range(len(rows)))
        ax.set_yticklabels([f"{est}{' (reported)' if est == REPORTED else ''}"
                            for est, _ in rows][::-1])
        ax.set_ylim(-0.6, len(rows) - 0.2)
        ax.set_xlabel(f"pooled std. coef ({X2_LABEL} | {X1_LABEL})")
        ax.set_title(f"{state} - one finding, three scalings")
        save_fig(fig, state, "three_scaling_forest")
        plt.show()

## Outputs

```
outputs/sequential_added_value/
  <state>/
    group_estimates.parquet      # one row per (subject, probe, structure)
    pooled_estimates.parquet     # one row per estimator x target (A, B, C)
    identity_checks.csv          # the max-abs-errors / violation counts from Validation
    epoch_sweep_pooled.parquet   # pooled estimates at every epoch length
    *.svg
```

In [ ]:
for state in STATES:
    d = OUTPUT_ROOT / state
    group_dfs[state].to_parquet(d / "group_estimates.parquet", index=False)
    pooled_dfs[state].to_parquet(d / "pooled_estimates.parquet", index=False)
    check_dfs[state].to_csv(d / "identity_checks.csv", index=False)
    sweep_df.query("state == @state").to_parquet(d / "epoch_sweep_pooled.parquet", index=False)
    print(f"{state}: wrote {sorted(p.name for p in d.iterdir())}")

## Reading the result

Group-by-group, to ~1e-15 here, `beta_A = beta_B * (1 - r12**2)` and
`beta_C = beta_B * sqrt(1 - r12**2)` (validation checks 2 and 7), so
`|beta_A| <= |beta_C| <= |beta_B|` (check 9) with a common sign (check 3). The three
estimators can never disagree about whether small-OFF area carries unique signal, only
about scale, and their p-values are near-identical by construction.

- Estimator A answers "how much of the delta left unexplained by medium/large OFF area
  moves with raw small-OFF area", and pays a `1 - r12**2` shrinkage for the part of `x2`
  that medium/large area already predicted.
- Estimator B answers "...moves with the excess small-OFF area", is unshrunk, and equals
  the joint `y ~ x1 + x2` coefficient on `x2` exactly (check 1).
- Estimator C is the same thing per SD of the regressor that actually enters the fit,
  which makes it the semipartial correlation (check 6); `beta_C**2` is the unique
  variance share (check 8).

## Reporting convention

Downstream text should quote one coefficient with one p-value. The convention here is
Estimator B, the conventional standardized partial coefficient, read as "change in `y`
per SD of `x2`, with `x1` held fixed". Quote Estimator C instead when the reader should
be on an explicitly variance-based footing, so that `beta_C**2` can be given as a share
of `y` variance; if so, keep the signed `beta_C` next to any squared value, since
squaring discards the sign. Avoid quoting `beta_A`: it is included because it is the
natural first thing to try, and because seeing where it sits relative to the other two
is informative, but its scaling has no standard name.

Quoting all three invites a reader to treat them as three findings, or to conclude the
result is scale-sensitive when it is not.

Extension caveat: with exactly two predictors, A and C shrink both coefficients by the
same factor, so relative comparisons between `x1` and `x2` survive. With three or more
the factor becomes `(1 - R2_i)`, different per predictor, and that no longer holds.

## Inference caveats

- Generated regressors. All three estimators feed stage-1 residuals into a later
  regression as observed data, so the reported standard errors do not propagate stage-1
  estimation error. With thousands of epochs per recording this should be negligible,
  but the SEs are not exact.
- Within-subject clustering. `(subject, probe, structure)` groups within a subject are
  not independent; the two-stage random-effects meta-analysis treats them as
  exchangeable. The fully-correct version is a 3-level mixed model with an AR(1)
  residual, which belongs in `r-offp` downstream.
- HAC across state gaps. Epochs are contiguous in state time, but removing the other
  states introduces wall-clock gaps, so lag-k is not exactly k x epoch in real time.
- Source. Full-48h `morphological` OFFs are the current best-but-provisional OFF source.
- Heterogeneity. I2 is high (>90%) throughout: the pooled number is a central tendency
  across recordings that genuinely differ, not a common effect.
